# 04. Trayectorias: categorizacion de rol y transiciones AA/DD

Proyecto de tesis (Maestria en Ciencia de Datos e IA, ESPOL): *Sistema de Generacion de Perfiles del Personal Docente y Administrativo en ESPOL para la asignacion inteligente de tareas*.

Este notebook agrega una fase nueva a la metodologia original (analisis longitudinal de trayectorias) a partir de `data/processed/historial_laboral_personas.csv` (ver notebook `01_preprocesamiento/10_historial_laboral_personas.ipynb`). No modifica esa fuente; solo la lee y agrega columnas/tablas derivadas.

Se ubica antes de `05_preparacion_modelado` porque sus features de trayectoria (tramos de rol, transiciones, antiguedad por categoria de rol) se incorporan al dataset que alimenta el clustering (`06_clustering`).

**Objetivo:** construir, a partir del historial de contratos de cada persona, una subdivision de rol mas fina que `TIPOEMPLEADO` (AA/DD) — `CATEGORIA_CARGO` — y detectar transiciones reales de rol (p.ej. AA -> DD) evitando falsos cambios causados por ruido administrativo o por contratos paralelos puntuales.

**Fuente:** `data/processed/historial_laboral_personas.csv`
**Funciones usadas (en `notebooks/01_preprocesamiento/_preprocesamiento_comun.py`):** `clasificar_categoria_cargo`, `construir_tramos_rol`, `detectar_transiciones_rol`, `extraer_eventos_puntuales`.
**Salidas:** `data/trayectorias/historial_laboral_categoria_cargo.csv`, `data/trayectorias/tramos_rol.csv`, `data/trayectorias/transiciones_rol.csv`, `data/trayectorias/eventos_puntuales_cargo.csv`.

**Decisiones de negocio detras de este notebook (confirmadas con el usuario, detalle en `context/DECISION_LOG.md`):**
1. La taxonomia `CATEGORIA_CARGO` se construye por separado dentro de cada rama de `TIPOEMPLEADO_DESC` (AA/DD nunca comparten reglas).
2. Los rotulos legado pre-reforma LOES sin la palabra "Titular" ("Profesor Agregado/Auxiliar/Principal") se fusionan en `DOCENTE_TITULAR_CARRERA` (confirmado por continuidad de persona).
3. Los rotulos legado ambiguos ("Profesor Pregrado", "Profesor", "Docente", "Docente(Grado)", todos anteriores a 2015) se resuelven por continuidad de persona hacia `DOCENTE_TITULAR_CARRERA` o `DOCENTE_NO_TITULAR_OCASIONAL`.
4. Los registros con `CARGO` tipo "Profesor/Docente..." pero `TIPOEMPLEADO_DESC`=ADMINISTRATIVO se tratan como ruido de captura de datos (`ANOMALIA_CARGO_DOCENTE_EN_AA`), no como senal de trayectoria.
5. Las categorias de contrato puntual/por proyecto (`CATEGORIAS_PUNTUALES`: servicios profesionales/proyecto, contratado por servicios civiles, actividad academica puntual desde administrativo, tribunal/comision, honorario/invitado) se excluyen del ESTADO de rol (no generan "transicion") y quedan como insumo de la futura capa de EVENTOS, porque pueden coexistir con el cargo estructural continuo de la persona.
6. `calcular_fecha_fin_efectiva` (usada por `construir_tramos_rol`) usa `ESTADOCONTRATO` (ACTIVO/POR EJECUTARSE = contrato abierto) para decidir vigencia, no solo la presencia de `FECHAFINCONTRATO` — ver DEC-005 en `context/DECISION_LOG.md` sobre el bug que esto corrige.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "01_preprocesamiento"))
import pandas as pd
import _preprocesamiento_comun as pc

PROJECT_ROOT = Path.cwd().parent.parent
TRAYECTORIAS_DIR = PROJECT_ROOT / "data" / "trayectorias"
TRAYECTORIAS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 100)


## 1. Carga del historial laboral ya procesado

Se parte de `data/processed/historial_laboral_personas.csv` (salida de `01_preprocesamiento/10_historial_laboral_personas.ipynb`), ya limpio y con `TIPOEMPLEADO_DESC` decodificado. Solo falta castear las fechas de contrato (el CSV las guarda como texto).


In [ ]:
df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "historial_laboral_personas.csv", low_memory=False)
df = pc.castear_fechas(df, ['FECHAINICIOCONTRATO', 'FECHAFINCONTRATO', 'FECHADESVINCULACION'])
print(f"{df.shape[0]} filas x {df.shape[1]} columnas, {df['IDPERSONA'].nunique()} personas")
df[['IDPERSONA', 'CARGO', 'TIPOEMPLEADO_DESC', 'FECHAINICIOCONTRATO', 'FECHAFINCONTRATO']].head()


## 2. Categorizacion de rol: `CATEGORIA_CARGO`

`pc.clasificar_categoria_cargo` agrega dos columnas:
- `CATEGORIA_CARGO`: subdivision de rol dentro de `TIPOEMPLEADO_DESC` (11 categorias en DD, 13 en AA â€” ver `pc.REGLAS_CATEGORIA_CARGO_DD` / `pc.REGLAS_CATEGORIA_CARGO_AA` para las reglas exactas).
- `ES_RUIDO_CALIDAD_DATOS`: `True` para `ANOMALIA_CARGO_DOCENTE_EN_AA` (cargo con apariencia docente pero tipificado como ADMINISTRATIVO) â€” se conserva para trazabilidad pero se excluye del analisis de transiciones.

Produce la jerarquia `TIPOEMPLEADO_DESC -> CATEGORIA_CARGO -> CARGO` (original, intacto).


In [ ]:
df = pc.clasificar_categoria_cargo(df)

for branch in ['DOCENTE', 'ADMINISTRATIVO']:
    sub = df[df['TIPOEMPLEADO_DESC'] == branch]
    print(f"--- {branch}: {len(sub)} filas ---")
    vc = sub['CATEGORIA_CARGO'].value_counts()
    print((vc / len(sub) * 100).round(1).astype(str) + '%  n=' + vc.astype(str))
    print()

print(f"ES_RUIDO_CALIDAD_DATOS=True: {df['ES_RUIDO_CALIDAD_DATOS'].sum()} filas ({df['ES_RUIDO_CALIDAD_DATOS'].mean()*100:.1f}%)")


## 3. Tramos de rol (ESTADO estructural) y eventos puntuales

`pc.construir_tramos_rol` colapsa contratos consecutivos de la misma `CATEGORIA_CARGO` en "tramos", igual que `pc.calcular_periodos_continuos` hace para la vinculacion general, pero **excluyendo** las categorias puntuales/por proyecto (`pc.CATEGORIAS_PUNTUALES`) y `ES_RUIDO_CALIDAD_DATOS`.

Estas categorias puntuales se extraen aparte con `pc.extraer_eventos_puntuales` (insumo para la futura capa de EVENTOS), porque pueden coexistir con el cargo estructural continuo de la persona (p.ej. un Docente Titular que ademas factura un contrato civil por un proyecto especifico) â€” incluirlas en el ESTADO generaba transiciones falsas (ver seccion 5).


In [ ]:
tramos_rol_bruto = pc.construir_tramos_rol(df)
eventos_puntuales = pc.extraer_eventos_puntuales(df)

# DEC-011: tramos_rol_bruto no modela concurrencia (ordena solo por fecha de inicio), asi
# que un nombramiento de autoridad ejercido en paralelo a un cargo estructural continuo
# (p.ej. Profesor Titular que es ademas Vicerrector) corta el cargo estructural en dos y
# genera transiciones de ida y vuelta que nunca ocurrieron. resolver_roles_simultaneos
# retira el tramo mas corto de cada solapamiento (el nombramiento de autoridad) y lo separa
# como "rol adicional simultaneo" -sin transicion-, dejando la linea principal de ESTADO
# limpia. Confirmado con el usuario: domina el tramo de mayor duracion.
tramos_rol, roles_adicionales_simultaneos = pc.resolver_roles_simultaneos(tramos_rol_bruto)

print(f"tramos_rol_bruto: {tramos_rol_bruto.shape[0]} filas, {tramos_rol_bruto['IDPERSONA'].nunique()} personas")
print(f"tramos_rol (tras resolver roles simultaneos): {tramos_rol.shape[0]} filas, {tramos_rol['IDPERSONA'].nunique()} personas")
print(f"roles_adicionales_simultaneos: {roles_adicionales_simultaneos.shape[0]} filas, "
      f"{roles_adicionales_simultaneos['IDPERSONA'].nunique()} personas")
print(f"eventos_puntuales: {eventos_puntuales.shape[0]} filas, {eventos_puntuales['IDPERSONA'].nunique()} personas")
tramos_rol.sort_values(['IDPERSONA', 'TRAMO_INICIO']).head(10)


## 4. Deteccion de transiciones de rol

`pc.detectar_transiciones_rol` recorre los tramos de cada persona y registra cada cambio de `CATEGORIA_CARGO` entre tramos consecutivos, con `TIPO_TRANSICION` a nivel `TIPOEMPLEADO` (AA_A_AA / AA_A_DD / DD_A_AA / DD_A_DD) y `GAP_DIAS` entre el fin del tramo de origen y el inicio del tramo de destino.


In [ ]:
transiciones_rol = pc.detectar_transiciones_rol(tramos_rol)  # ya sobre la linea limpia (DEC-011)

print(f"{transiciones_rol.shape[0]} transiciones detectadas en {transiciones_rol['IDPERSONA'].nunique()} personas")
print()
print("Por tipo (a nivel TIPOEMPLEADO):")
print(transiciones_rol['TIPO_TRANSICION'].value_counts())
print()
print("Transiciones por persona (distribucion):")
print(transiciones_rol.groupby('IDPERSONA').size().describe())
print()
print("Pares CATEGORIA_ORIGEN -> CATEGORIA_DESTINO mas frecuentes para AA_A_DD:")
sub = transiciones_rol[transiciones_rol['TIPO_TRANSICION'] == 'AA_A_DD']
print((sub['CATEGORIA_ORIGEN'] + ' -> ' + sub['CATEGORIA_DESTINO']).value_counts().head(10))
print()
print("Pares mas frecuentes para DD_A_AA:")
sub = transiciones_rol[transiciones_rol['TIPO_TRANSICION'] == 'DD_A_AA']
print((sub['CATEGORIA_ORIGEN'] + ' -> ' + sub['CATEGORIA_DESTINO']).value_counts().head(10))


## 5. Limitacion conocida: encargos cortos / interinatos

Excluir las categorias puntuales (seccion 3) redujo las transiciones de 11,854 a ~4,468 y el maximo de transiciones por persona de 103 a 49. El caso remanente con mas transiciones (49) no es ruido de contratos paralelos, sino un patron distinto: una persona con cargo estructural continuo (Docente Titular) que ademas asume repetidamente nombramientos cortos de autoridad (varios de 1 a 10 dias) â€” un patron de **encargos/interinatos** (p.ej. cubrir unos dias como autoridad mientras el titular del cargo esta de viaje), no un cambio real y sostenido de rol.

No se resuelve en este notebook porque requiere una decision de negocio (p.ej. un umbral minimo de duracion de tramo para contar como transicion real, o tratar los encargos cortos de autoridad como su propia categoria de evento en vez de un cambio de ESTADO) que aun no se ha definido con el usuario. Se documenta aqui para la siguiente iteracion.


In [ ]:
# Diagnostico: distribucion de duracion de tramo para transiciones (dias), y
# el caso con mas transiciones, para ilustrar el patron de encargos cortos.
transiciones_rol['GAP_DIAS_NUM'] = pd.to_numeric(transiciones_rol['GAP_DIAS'], errors='coerce')
print("GAP_DIAS (todas las transiciones):")
print(transiciones_rol['GAP_DIAS_NUM'].describe())
transiciones_rol = transiciones_rol.drop(columns=['GAP_DIAS_NUM'])

top_persona = transiciones_rol.groupby('IDPERSONA').size().idxmax()
print(f"\nPersona con mas transiciones: IDPERSONA={top_persona}")
tramos_rol[tramos_rol['IDPERSONA'] == top_persona].sort_values('TRAMO_INICIO')[
    ['CATEGORIA_CARGO', 'TRAMO_INICIO', 'TRAMO_FIN', 'N_CONTRATOS']
].head(15)


## 6. Guardado en `data/trayectorias/`

Se guardan cuatro tablas nuevas, separadas de `data/processed/` (que sigue siendo responsabilidad de `01_preprocesamiento/10_historial_laboral_personas.ipynb`) para no interferir con ese pipeline:

- `historial_laboral_categoria_cargo.csv`: el historial de contratos completo con `CATEGORIA_CARGO`/`ES_RUIDO_CALIDAD_DATOS` agregadas.
- `tramos_rol.csv`: tramos de rol estructural por persona (ESTADO).
- `transiciones_rol.csv`: transiciones detectadas entre tramos.
- `eventos_puntuales_cargo.csv`: contratos de categorias puntuales/por proyecto (insumo de la futura capa de EVENTOS).


In [ ]:
cols_historial_export = [
    'IDPERSONA', 'IDCONTRATOLABORAL', 'CARGO', 'TIPOEMPLEADO_DESC', 'CATEGORIA_CARGO',
    'ES_RUIDO_CALIDAD_DATOS', 'FECHAINICIOCONTRATO', 'FECHAFINCONTRATO', 'FECHADESVINCULACION',
]
df[cols_historial_export].to_csv(
    TRAYECTORIAS_DIR / 'historial_laboral_categoria_cargo.csv', index=False, encoding='utf-8-sig'
)
tramos_rol.to_csv(TRAYECTORIAS_DIR / 'tramos_rol.csv', index=False, encoding='utf-8-sig')
transiciones_rol.to_csv(TRAYECTORIAS_DIR / 'transiciones_rol.csv', index=False, encoding='utf-8-sig')
eventos_puntuales.to_csv(TRAYECTORIAS_DIR / 'eventos_puntuales_cargo.csv', index=False, encoding='utf-8-sig')
roles_adicionales_simultaneos.to_csv(TRAYECTORIAS_DIR / 'roles_adicionales_simultaneos.csv', index=False, encoding='utf-8-sig')

for nombre in ['historial_laboral_categoria_cargo.csv', 'tramos_rol.csv', 'transiciones_rol.csv', 'eventos_puntuales_cargo.csv', 'roles_adicionales_simultaneos.csv']:
    ruta = TRAYECTORIAS_DIR / nombre
    print(f"Guardado: {ruta}")


## 6b. Registro de autoridades / funciones adicionales (DEC-012)

`registro_autoridades.csv` (limpiado en `01_preprocesamiento/19_registro_autoridades.ipynb`)
registra cargos/designaciones ejercidos **en paralelo** al contrato laboral (Director,
Coordinador, membresias de consejo, subrogaciones) - no reemplaza `CATEGORIA_CARGO_ACTUAL`
ni genera tramos/transiciones. `pc.procesar_registro_autoridades` lo enriquece comparando
cada designacion contra el contrato vigente en `historial_laboral` en esas mismas fechas
(`ES_COINCIDENTE_CONTRATO`), y `pc.construir_features_funciones_adicionales` lo resume por
persona. Confirmado con el usuario: una subrogacion NUNCA se trata como transicion
contractual (sin importar su duracion), se conservan TODAS las subrogaciones, y los
registros de "Miembro del consejo directivo/politecnico - estudiantes" se conservan pero
marcados con `ES_REPRESENTANTE_ESTUDIANTIL=True` (98% no tienen ningun contrato de personal
solapado - son casi con certeza representantes estudiantiles, no poblacion del proyecto).

In [ ]:
registro_autoridades = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "registro_autoridades.csv",
    parse_dates=["FECHADESDE", "FECHAHASTA"],
)
funciones_adicionales = pc.procesar_registro_autoridades(registro_autoridades, df)
features_funciones_adicionales = pc.construir_features_funciones_adicionales(funciones_adicionales)

print(f"funciones_adicionales: {funciones_adicionales.shape[0]} filas, "
      f"{funciones_adicionales['IDPERSONA'].nunique()} personas")
print(f"  ES_REPRESENTANTE_ESTUDIANTIL: {funciones_adicionales['ES_REPRESENTANTE_ESTUDIANTIL'].sum()}")
print(f"  ES_RUIDO_CALIDAD_DATOS: {funciones_adicionales['ES_RUIDO_CALIDAD_DATOS'].sum()}")
print(f"  ES_DUPLICADO_EXACTO: {funciones_adicionales['ES_DUPLICADO_EXACTO'].sum()}")
print(f"  ES_SUBROGACION: {funciones_adicionales['ES_SUBROGACION'].sum()}")
print()
print(f"features_funciones_adicionales: {features_funciones_adicionales.shape[0]} personas con alguna funcion valida")
print("TUVO_FUNCION_NO_COINCIDENTE_CON_CONTRATO:",
      features_funciones_adicionales['TUVO_FUNCION_NO_COINCIDENTE_CON_CONTRATO'].value_counts().to_dict())

funciones_adicionales.to_csv(TRAYECTORIAS_DIR / 'funciones_adicionales_persona.csv', index=False, encoding='utf-8-sig')
print(f"Guardado: {TRAYECTORIAS_DIR / 'funciones_adicionales_persona.csv'}")
funciones_adicionales.head()

## 6c. Eventos unificados de trayectoria (DEC-014)

`pc.construir_eventos_trayectoria` combina en una sola linea de tiempo por persona los
tramos de rol dentro de ESPOL (`tramos_rol`), las funciones adicionales/subrogaciones
(`funciones_adicionales`, excluyendo las que coinciden con el contrato vigente) y la
experiencia laboral externa previa/paralela (`experiencia_externa.csv`). Es una tabla
nueva, puramente derivada de datos ya construidos aqui - no cambia `tramos_rol.csv`,
`funciones_adicionales_persona.csv` ni `features_trayectoria_persona.csv`, y no se usa
para clustering. Pensada para reutilizarse tanto en la construccion del documento
semantico (`07_embeddings.ipynb`) como en la vista de trayectoria del dashboard
(`08_dashboard`), evitando reconstruir la misma linea de tiempo dos veces.

In [ ]:
experiencia_externa = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "experiencia_externa.csv", low_memory=False,
)
# No se usa parse_dates de read_csv: al menos una fecha mal formada en este archivo crudo
# (nunca paso por pc.parsear_fecha) hace que pandas descarte el parseo de TODA la columna
# silenciosamente y la deje como texto - pc.castear_fechas (pd.to_datetime con
# errors="coerce" y rango de años valido) es robusto a eso.
experiencia_externa = pc.castear_fechas(experiencia_externa, ["FECHADESDE", "FECHAHASTA"])

# DEC-018: se incluyen los contratos puntuales/por proyecto dentro de ESPOL
# (eventos_puntuales, seccion 3) - antes NO aparecian en ningun lado de la linea de
# tiempo (ni en tramos_rol, excluidos por diseno desde DEC-004, ni en eventos_puntuales
# de esta funcion), lo que podia mostrar a una persona como si hubiera dejado de
# trabajar en la fecha de su ultimo tramo ESTRUCTURAL aunque su contrato puntual
# siguiera vigente (ver DEC-018 en context/DECISION_LOG.md).
# DEC-022: mapeo SIGLAS<->NOMBRE desde registro_autoridades (ya cargado en la seccion
# 6a) - se agrega la sigla real (p.ej. "(GTSI)") junto al nombre completo de la unidad
# en el texto, porque el modelo de embeddings no sabe que son la misma entidad
# (similitud coseno GTSI vs nombre completo ~0.79, igual al ruido de anisotropia entre
# textos NO relacionados) - sin esto, una busqueda con la sigla no encontraba a nadie.
mapa_siglas_unidad = pc.construir_mapa_siglas_unidad(registro_autoridades)

eventos_trayectoria = pc.construir_eventos_trayectoria(
    tramos_rol, funciones_adicionales, experiencia_externa, eventos_puntuales,
    mapa_siglas_unidad,
)

print(f"eventos_trayectoria: {eventos_trayectoria.shape[0]} filas, "
      f"{eventos_trayectoria['IDPERSONA'].nunique()} personas")
print(eventos_trayectoria["TIPO_EVENTO"].value_counts().to_string())
print("dtype FECHA_INICIO:", eventos_trayectoria["FECHA_INICIO"].dtype)

eventos_trayectoria.to_csv(TRAYECTORIAS_DIR / "eventos_trayectoria_persona.csv", index=False, encoding="utf-8-sig")
print(f"Guardado: {TRAYECTORIAS_DIR / 'eventos_trayectoria_persona.csv'}")
eventos_trayectoria.head()

## 7. Features de trayectoria por persona (para clustering)

`pc.construir_features_trayectoria` agrega `tramos_rol` y `transiciones_rol` a una fila por `IDPERSONA`, con la granularidad que necesita `05_preparacion_modelado` para fusionarlas al dataset de modelado (ver DEC-006 en `context/DECISION_LOG.md`).

Personas sin ningún tramo de rol estructural (solo tuvieron contratos de categorías puntuales) no aparecen aquí — `05_preparacion_modelado` decide cómo tratar esos nulos al fusionar.


In [ ]:
features_trayectoria = pc.construir_features_trayectoria(tramos_rol, transiciones_rol, roles_adicionales_simultaneos)

# DEC-012: fusionar el resumen de funciones adicionales (registro_autoridades) - fuente
# distinta y complementaria a roles_adicionales_simultaneos (DEC-011, inferido de solapes
# de contrato); no se reemplazan entre si.
features_trayectoria = features_trayectoria.merge(features_funciones_adicionales, on='IDPERSONA', how='left')
features_trayectoria['TUVO_FUNCION_ADICIONAL'] = features_trayectoria['TUVO_FUNCION_ADICIONAL'].fillna(False)
features_trayectoria['TUVO_SUBROGACION'] = features_trayectoria['TUVO_SUBROGACION'].fillna(False)
features_trayectoria['TUVO_FUNCION_NO_COINCIDENTE_CON_CONTRATO'] = features_trayectoria['TUVO_FUNCION_NO_COINCIDENTE_CON_CONTRATO'].fillna(False)
for c in ['N_FUNCIONES_ADICIONALES', 'N_CATEGORIAS_FUNCION_ADICIONAL_DISTINTAS', 'N_SUBROGACIONES']:
    features_trayectoria[c] = features_trayectoria[c].fillna(0).astype(int)

# DEC-017 (ver context/DECISION_LOG.md): personas cuyo UNICO contrato vigente hoy es de
# categoria puntual (p.ej. CONTRATO_SERVICIOS_PROFESIONALES_PROYECTO) no tienen tramo
# estructural activo - CATEGORIA_CARGO_ACTUAL/CLUSTER siguen correctamente basados en su
# ultimo tramo ESTRUCTURAL (no cambia), pero CARGO_ACTUAL/UNIDAD_ACTUAL_NOMBRE (uso de
# presentacion en el dashboard) deben reflejar el contrato puntual vigente cuando exista,
# no un cargo estructural ya finalizado - de lo contrario VIGENTE_ACTUALMENTE=True
# contradice visualmente un cargo que parece haber terminado hace mas de un año.
contrato_puntual_vigente = pc.construir_contrato_puntual_vigente(df)
features_trayectoria = features_trayectoria.merge(contrato_puntual_vigente, on='IDPERSONA', how='left')

features_trayectoria.to_csv(TRAYECTORIAS_DIR / 'features_trayectoria_persona.csv', index=False, encoding='utf-8-sig')

print(f"Guardado: {TRAYECTORIAS_DIR / 'features_trayectoria_persona.csv'} "
      f"({features_trayectoria.shape[0]} filas x {features_trayectoria.shape[1]} columnas)")
print()
print("ES_TRAYECTORIA_ESTABLE:", features_trayectoria['ES_TRAYECTORIA_ESTABLE'].value_counts().to_dict())
print("TUVO_TRANSICION_AA_A_DD:", features_trayectoria['TUVO_TRANSICION_AA_A_DD'].value_counts().to_dict())
print("TUVO_TRANSICION_DD_A_AA:", features_trayectoria['TUVO_TRANSICION_DD_A_AA'].value_counts().to_dict())
features_trayectoria.head()
